### **Latent Space Analysis and Visualisation**

In [1]:
import sys
from typing import Any

from tqdm import tqdm
import torch
from torch.types import Tensor


def get_size(obj: object, default: Any = -1) -> int:
    """Computes how much memory storage is being used by input obj in [bytes]."""
    return sys.getsizeof(obj, default=default)

In [2]:
#BASEPATH: str = '/home/edoardo/Desktop/MockDataForDMs'
BASEPATH: str = '/mnt/d/MockDataForDMs'

latent_dmap: dict[int, Tensor] = torch.load(f'{BASEPATH}/latentSpaceContainer.pt')

print(
    f'Latent Space dmap size: {get_size(latent_dmap)} bytes\n'
    f'Number of epochs registered: {len(latent_dmap)}\n'
    f'Embedded data size: {latent_dmap[0].shape}\n'
)

# NOTE:
#   - a tensor of shape [E, B, C, H, W] occupies less memory than the latent dict map
#   - when saved though, the 5D tensor occupies x2 space (dmap: ~13.7MB, tensor: ~27  MB)
#   - this has been tested on 5 epochs (TODO: test if for larger # of epochs is the same)

Latent Space dmap size: 224 bytes
Number of epochs registered: 5
Embedded data size: torch.Size([800, 8, 15, 15])



In [3]:
from itertools import islice
import numpy as np
from numpy.typing import NDArray


def manage_embedded_data(latent_dmap: dict[int, Tensor]) -> dict[int, NDArray]:
    """
    Takes the data in the latent dmap and for each epoch flattens the respective
    embedded vector of shape [B, C, H, W] to a tensor [B, C x H x W], finally
    converting it to a numpy array.
    """
    # use `islice` bc `latent_dmap` could be very large (epochs
    # number + respective tensors with all the latent vectors.)
    flattened_dmap: dict[int, NDArray] = {
        epoch: emb.flatten(1, -1).numpy()
        for epoch, emb in islice(latent_dmap.items(), len(latent_dmap))
    }
    return flattened_dmap

In [4]:
np_emb = manage_embedded_data(latent_dmap)